In [1]:
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
import pickle
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz

import sys
sys.path.append("../../")
from testing_scripts.qaoa_utils import QAOASolver, evaluate_energy

import numpy as np
import pcbo_utils
from qiskit.quantum_info import SparsePauliOp

In [2]:
n_qubits = 14
feature_set, feature_to_idx, first_corr_arr, second_corr_arr, third_corr_arr = pcbo_utils.load_features_and_corr_files(f'sampled_{n_qubits}_features_subproblem_1 copy')

pcbo_obj = pcbo_utils.create_three_body_cubo(
    feature_set,
    first_corr_arr,
    second_corr_arr,
    third_corr_arr,
    feature_to_idx,
    select_n_features=4,
)

pubo = {key: float(value) for key, value in pcbo_obj.to_pubo().items()}

In [3]:
def convert_pubo_to_ising(hypergraph: dict) -> list[tuple[str, float]]:
    """Convert a hypergraph dictionary to a list of Pauli strings with weights."""
    n = n_qubits # Number of qubits
    pauli_list = []

    for edge, weight in hypergraph.items():
        if edge:  # Ensure the edge is not empty
            # Create a Pauli string with "I" for all qubits
            paulis = ["I"] * n
            # Replace "I" with "Z" for qubits in the edge
            for node in edge:
                paulis[node] = "Z"
            # Append the reversed Pauli string and weight to the list
            pauli_list.append(("".join(paulis[::-1]), weight))

    return pauli_list

max_cut_paulis = convert_pubo_to_ising(pubo)
cost_hamiltonian = SparsePauliOp.from_list(max_cut_paulis)
paulis,coeffs = cost_hamiltonian.paulis.to_labels(),cost_hamiltonian.coeffs.real
cost_hamiltonian

SparsePauliOp(['IIIIIIIIIIIIIZ', 'IIIIIIIIIIIIZI', 'IIIIIIIIIIIZII', 'IIIIIIIIIIZIII', 'IIIIIIIIIZIIII', 'IIIIIIIIZIIIII', 'IIIIIIIZIIIIII', 'IIIIIIZIIIIIII', 'IIIIIZIIIIIIII', 'IIIIZIIIIIIIII', 'IIIZIIIIIIIIII', 'IIZIIIIIIIIIII', 'IZIIIIIIIIIIII', 'ZIIIIIIIIIIIII', 'IIIIIIIIIIIIZZ', 'IIIIIIIIIIIZIZ', 'IIIIIIIIIIZIIZ', 'IIIIIIIIIZIIIZ', 'IIIIIIIIZIIIIZ', 'IIIIIIIZIIIIIZ', 'IIIIIIZIIIIIIZ', 'IIIIIZIIIIIIIZ', 'IIIIZIIIIIIIIZ', 'IIIZIIIIIIIIIZ', 'IIZIIIIIIIIIIZ', 'IZIIIIIIIIIIIZ', 'ZIIIIIIIIIIIIZ', 'IIIIIIIIIIIZZI', 'IIIIIIIIIIZIZI', 'IIIIIIIIIZIIZI', 'IIIIIIIIZIIIZI', 'IIIIIIIZIIIIZI', 'IIIIIIZIIIIIZI', 'IIIIIZIIIIIIZI', 'IIIIZIIIIIIIZI', 'IIIZIIIIIIIIZI', 'IIZIIIIIIIIIZI', 'IZIIIIIIIIIIZI', 'ZIIIIIIIIIIIZI', 'IIIIIIIIIIZZII', 'IIIIIIIIIZIZII', 'IIIIIIIIZIIZII', 'IIIIIIIZIIIZII', 'IIIIIIZIIIIZII', 'IIIIIZIIIIIZII', 'IIIIZIIIIIIZII', 'IIIZIIIIIIIZII', 'IIZIIIIIIIIZII', 'IZIIIIIIIIIZII', 'ZIIIIIIIIIIZII', 'IIIIIIIIIZZIII', 'IIIIIIIIZIZIII', 'IIIIIIIZIIZIII', 'IIIIIIZIIIZIII', 'IIIIIZIIIIZI

In [4]:
reps=2
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=reps)

In [5]:
teague_qaoa = QAOASolver(cost_hamiltonian,circuit,"CPU")
teague_qaoa.prepare_circuit()
teague_qaoa.err = None
teague_qaoa.pcirc.num_parameters

966

In [6]:
# Solve with classical Eigensolver for comparison
exact_solution = teague_qaoa.evaluate_exact_energy()

Exact Energy from Eigensolver: -26014.186972876967


In [7]:
# #Importing from seperate file 
# with open(f"teague_pickle_data/{n_qubits}_qb_teague_cafqa_results.pkl", "rb") as f:
#     cafqa_data = pickle.load(f)
# best_cafqa_fitness_values = cafqa_data["best_cafqa_fitness_values"]
# best_cafqa_parameters = cafqa_data["best_cafqa_parameters"]
# print("cafqa_data keys:", list(cafqa_data.keys()))
# teague_qaoa.energy_best = cafqa_data['CAFQA_initialization_energy']
# teague_qaoa.ks_best = best_cafqa_parameters[0]

In [8]:
# we can perform CAFQA by using the main optimization function "claptonize"
teague_qaoa.run_cafqa(n_gens=1)

STARTING ROUND 0


started GA at id 1 with 8 procs

started GA at id 2 with 8 procs


started GA at id 3 with 8 procs
started GA at id None with 8 procs

GA parameters used for this experiment:
  num_generations=5
  num_parents_mating=20
  population_size=100
  num_genes=966
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5
GA parameters used for this experiment:
  num_generations=5
  num_parents_mating=20
  population_size=100
  num_genes=966
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5
GA parameters used for this experiment:
  num_generations=5
  num_parents_mating=20
  population_size=100
  num_genes=966
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=ada

In [9]:
teague_qaoa.energy_best

np.float64(-1999.285400390625)

In [10]:
# cafqa_angles = [param * np.pi/2 for param in teague_qaoa.ks_best]

custom = [3, 0, 2, 2, 2, 1, 2, 0, 1, 0, 0, 1, 3, 1, 0, 3, 3, 3, 1, 0, 0, 1,
       1, 3, 3, 3, 1, 0, 2, 3, 0, 3, 3, 1, 3, 3, 1, 0, 3, 2, 2, 3, 2, 0,
       1, 2, 0, 3, 2, 2, 3, 2, 3, 0, 0, 0, 0, 0, 3, 3, 3, 0, 0, 2, 3, 3,
       3, 3, 1, 3, 3, 0, 0, 2, 2, 1, 1, 0, 1, 3, 2, 1, 0, 3, 3, 3, 0, 3,
       0, 2, 1, 2, 2, 3, 1, 1, 3, 0, 0, 1, 0, 0, 3, 1, 1, 0, 2, 2, 3, 3,
       0, 2, 3, 2, 2, 0, 2, 2, 2, 1, 1, 2, 0, 0, 0, 2, 1, 1, 0, 3, 2, 2,
       3, 3, 2, 0, 3, 2, 0, 0, 0, 1, 3, 0, 0, 0, 0, 1, 1, 1, 2, 0, 2, 0,
       2, 0, 2, 2, 1, 1, 0, 2, 0, 0, 3, 3, 3, 2, 3, 1, 2, 3, 3, 1, 1, 0,
       0, 3, 3, 3, 0, 3, 1, 2, 0, 2, 3, 0, 0, 1, 2, 1, 3, 0, 2, 1, 3, 3,
       0, 1, 2, 1, 1, 2, 3, 1, 3, 3, 0, 2, 3, 3, 0, 0, 1, 3, 2, 2, 2, 0,
       3, 3, 3, 3, 2, 2, 3, 1, 3, 0, 2, 1, 1, 2, 2, 0, 3, 3, 3, 3, 3, 1,
       3, 2, 0, 0, 1, 3, 3, 2, 3, 1, 3, 3, 2, 3, 2, 1, 3, 0, 3, 3, 1, 3,
       3, 3, 1, 3, 3, 1, 2, 3, 3, 3, 2, 0, 2, 2, 3, 3, 3, 1, 0, 1, 1, 2,
       3, 2, 2, 0, 1, 0, 3, 3, 0, 1, 1, 0, 2, 2, 1, 1, 0, 1, 1, 1, 0, 3,
       3, 3, 3, 0, 3, 0, 3, 1, 2, 1, 3, 3, 3, 1, 0, 0, 2, 1, 2, 0, 3, 0,
       2, 0, 1, 3, 2, 0, 3, 3, 0, 2, 2, 3, 1, 0, 2, 3, 3, 1, 0, 0, 0, 3,
       0, 0, 0, 0, 1, 3, 3, 1, 0, 1, 1, 0, 1, 3, 1, 1, 3, 2, 2, 1, 0, 1,
       0, 2, 2, 1, 2, 1, 3, 0, 1, 0, 3, 1, 0, 3, 0, 1, 1, 0, 0, 2, 0, 2,
       2, 0, 3, 2, 3, 1, 1, 2, 0, 1, 0, 0, 2, 0, 2, 3, 1, 1, 3, 3, 1, 0,
       2, 2, 2, 3, 0, 3, 1, 2, 3, 2, 2, 2, 3, 3, 2, 3, 1, 0, 1, 2, 1, 1,
       0, 2, 2, 3, 3, 0, 0, 2, 2, 0, 2, 1, 0, 0, 2, 3, 0, 2, 0, 1, 0, 2,
       0, 1, 3, 1, 1, 2, 1, 1, 1, 0, 2, 1, 0, 0, 2, 1, 0, 2, 3, 3, 0, 1,
       3, 2, 1, 0, 2, 3, 3, 2, 3, 0, 0, 1, 1, 1, 2, 3, 1, 0, 1, 0, 3, 2,
       3, 0, 3, 0, 1, 1, 0, 0, 3, 3, 0, 3, 1, 2, 1, 3, 1, 3, 2, 2, 0, 0,
       3, 0, 0, 2, 0, 0, 1, 2, 3, 2, 2, 2, 1, 1, 0, 0, 1, 2, 2, 0, 2, 3,
       3, 3, 2, 0, 3, 2, 0, 1, 2, 1, 1, 1, 0, 3, 0, 1, 2, 2, 2, 1, 1, 3,
       1, 0, 0, 2, 3, 1, 3, 0, 0, 0, 1, 0, 1, 3, 0, 3, 0, 2, 2, 0, 3, 2,
       3, 1, 2, 3, 0, 0, 3, 1, 2, 2, 2, 0, 1, 3, 3, 0, 1, 1, 0, 0, 3, 1,
       0, 2, 3, 1, 1, 0, 0, 3, 3, 2, 3, 3, 0, 1, 2, 0, 3, 0, 3, 0, 0, 2,
       1, 3, 1, 0, 1, 2, 2, 0, 3, 0, 0, 3, 1, 1, 3, 2, 1, 1, 0, 3, 3, 1,
       1, 2, 1, 3, 2, 2, 3, 3, 2, 3, 3, 0, 2, 1, 3, 1, 2, 3, 3, 2, 3, 3,
       2, 3, 1, 0, 2, 0, 2, 3, 0, 2, 1, 2, 1, 1, 2, 2, 1, 3, 3, 3, 2, 0,
       2, 2, 3, 2, 2, 1, 1, 2, 2, 1, 3, 3, 2, 3, 0, 2, 2, 1, 1, 0, 0, 3,
       3, 0, 0, 0, 1, 1, 3, 1, 1, 1, 0, 3, 3, 3, 1, 3, 2, 2, 2, 3, 0, 0,
       2, 0, 3, 2, 0, 0, 0, 2, 1, 2, 3, 3, 2, 2, 0, 2, 1, 1, 1, 3, 3, 2,
       2, 2, 0, 1, 0, 2, 0, 2, 1, 2, 3, 1, 0, 1, 1, 1, 3, 1, 2, 0, 2, 3,
       0, 3, 1, 0, 3, 0, 0, 2, 1, 0, 1, 2, 2, 2, 2, 2, 2, 3, 2, 0, 0, 1,
       1, 0, 1, 0, 1, 1, 2, 0, 3, 2, 3, 3, 3, 1, 2, 0, 0, 2, 3, 1, 0, 2,
       1, 3, 1, 3, 2, 0, 3, 3, 3, 2, 3, 2, 2, 3, 2, 1, 3, 1, 1, 2, 1, 0,
       0, 0, 0, 1, 0, 2, 0, 2, 2, 0, 1, 0, 0, 2, 1, 1, 1, 3, 1, 2, 0, 0,
       1, 0, 3, 0, 0, 3, 2, 1, 3, 1, 0, 1, 0, 0, 2, 2, 3, 3, 0, 3, 3, 0,
       3, 0, 0, 1, 2, 2, 1, 0, 1, 3, 1, 2, 2, 2, 0, 1, 1, 1, 2, 2, 1, 2,
       3, 2, 1, 3, 1, 1, 3, 0, 0, 3, 2, 0, 2, 2, 3, 1, 1, 0, 0, 2, 0, 0,
       2, 1, 2, 3, 1, 3, 1, 0, 1, 0, 1, 2, 3, 0, 1, 2, 1, 2, 3, 0]

cafqa_angles = [param * np.pi/2 for param in custom]


In [11]:
energies = [teague_qaoa.evaluate_energy(teague_qaoa.pcirc,teague_qaoa.cost_hamiltonian,cafqa_angles) for _ in range(10)]
average_energy = np.mean(energies)
print(f"Average CAFQA Qiskit Energy: {average_energy}")

Average CAFQA Qiskit Energy: -7000.492529594079


In [ ]:
# Random Initalization
random_angles = np.random.random(len(teague_qaoa.ks_best))
random_energies = [evaluate_energy(teague_qaoa.pcirc, cost_hamiltonian, random_angles) for _ in range(10)]
min_energy = min(random_energies)
print(f"Minimum Energy found with Random initialization over 100 runs: {min_energy}")

In [ ]:
cafqa_result,cafqa_iteration_vals = teague_qaoa.run_qaoa(cafqa_angles,max_iters=1000)
random_result,random_iteration_vals = teague_qaoa.run_qaoa(random_angles,max_iters=1000)

In [ ]:
cafqa_result

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(cafqa_iteration_vals, label="CAFQA")
plt.plot(random_iteration_vals, label="Random Initialization")
plt.legend()
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()